In [51]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style('whitegrid')
import warnings
warnings.filterwarnings('ignore')
import duckdb

In [52]:
# Carregar os datasets CSV da pasta 'data'
customers = pd.read_csv('../data/olist_customers_dataset.csv')
geolocation = pd.read_csv('../data/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/olist_orders_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
product_translation = pd.read_csv('../data/product_category_name_translation.csv')

In [269]:
query= """
WITH tb_atraso AS (
    SELECT DISTINCT order_id,
            customer_id,
            order_status,
            date(substr(order_delivered_customer_date, 1, 10)) - date(substr(order_estimated_delivery_date, 1, 10)) AS DtDias
    FROM orders
    WHERE order_status = 'delivered'
),

tb_new_atraso AS (

SELECT *,
        CASE
            WHEN DtDias <= 0 THEN 0 
            WHEN DtDias > 0 THEN 1
        END AS DtAtraso,
        CASE
            WHEN DtDias <= 0 THEN 'sem_atraso'
            WHEN DtDias <= 3 THEN '1-3_dias_atraso'
            WHEN DtDias <= 7 THEN '1-7_dias_atraso'
            ELSE '7+_dias_atraso'
        END AS DtDiasAtraso
FROM tb_atraso  
)

SELECT *
FROM tb_new_atraso



"""

data = duckdb.query(query=query).to_df()

In [271]:
data.head(2)

,order_id,customer_id,order_status,DtDias,DtAtraso,DtDiasAtraso
0,3faf126d8a3b0684272791cd3c6cf3fe,824841f5112b77c04e832f5a46c2d733,delivered,-21,0,sem_atraso
1,ec3d63b91b19bedcbe453ea0df14dd50,379792aa619482a6545bd1db3905ab2b,delivered,-11,0,sem_atraso


#Tempo de entrega

In [270]:
data

,order_id,customer_id,order_status,DtDias,DtAtraso,DtDiasAtraso
0,3faf126d8a3b0684272791cd3c6cf3fe,824841f5112b77c04e832f5a46c2d733,delivered,-21,0,sem_atraso
1,ec3d63b91b19bedcbe453ea0df14dd50,379792aa619482a6545bd1db3905ab2b,delivered,-11,0,sem_atraso
2,31426d292a5d34315a18ac11d6eef7eb,bf4fe999a2d7bc0465cbd817fac3bbee,delivered,-19,0,sem_atraso
3,1b5bd1070a7857f30e0856e204f30f25,e30b71f331d7484590d41e75507583fd,delivered,-4,0,sem_atraso
4,6f36c999f8ef8d1a3a999079a5b637aa,ac055528f5a69cba5e02d73ad1ee9a74,delivered,1,1,1-3_dias_atraso
...,...,...,...,...,...,...
96473,e0cb1558bbe670b4cd84bf99bf93318f,956d144703c3b72fcb05e587e39f3ab2,delivered,-18,0,sem_atraso
96474,9058bc799c3791077efb3aaa8e0d5e02,3fb83e207757e8f15ff70d937c86929b,delivered,-30,0,sem_atraso
96475,0fa1fab1d7c1211c824596ed5e111e3c,7f3bd6c94d2daf7b6462d1a894a775b4,delivered,3,1,1-3_dias_atraso
96476,add4f182072426430ee6c993eab97efe,b87639f5efd3e2316dca5dec5e2f88f4,delivered,-10,0,sem_atraso


In [85]:
data['DtAtraso'].value_counts()

DtAtraso
0    89936
1     6534
Name: count, dtype: Int64

In [86]:
data['DtDiasAtraso'].value_counts().sort_values(ascending=False)

DtDiasAtraso
sem_atraso         89936
7+_dias_atraso      2870
1-3_dias_atraso     1870
1-7_dias_atraso     1802
Name: count, dtype: int64

In [101]:
print(f'Pedidos sem atraso: {round((data['DtDiasAtraso'] == 'sem_atraso').mean() * 100, 2)}%')
print(f'Pedidos entre 1 a 3 dias de atraso: {round((data['DtDiasAtraso'] == '1-3_dias_atraso').mean() * 100, 2)}%')
print(f'Pedidos entre 4 a 7 dias de atraso: {round((data['DtDiasAtraso'] == '1-7_dias_atraso').mean() * 100, 2)}%')


Pedidos sem atraso: 93.22%
Pedidos entre 1 a 3 dias de atraso: 1.94%
Pedidos entre 4 a 7 dias de atraso: 1.87%


In [103]:
duckdb.register('data', data)
print("Tabela registrada")

Tabela registrada


In [ ]:
query = """

SELECT d.*,
        review_id,
        review_score
FROM data AS d
INNER JOIN order_reviews AS r
ON d.order_id = r.order_id
"""


data_01 = duckdb.query(query=query).to_df()


In [122]:
data_01

,order_id,customer_id,order_status,DtDias,DtAtraso,DtDiasAtraso,review_id,review_score
0,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,-7,0,sem_atraso,abc5655186d40772bd6e410420e6a3ed,5
1,203096f03d82e0dffbc41ebc2e2bcfb7,d2b091571da224a1b36412c18bc3bbfe,delivered,11,1,7+_dias_atraso,38cae21b1b57a95959440380d5b2ef7a,2
2,775130c792f4c6460c63162e994a1114,d4cd9729eeca5306a2c039fe6e55601e,delivered,-24,0,sem_atraso,a35cca2abb31df8bd9c12fab7b28b386,5
3,a0d5b8474423ddf55228373b81a46272,3f7d26944f7f68bd2ac23b5e8b500ab0,delivered,-16,0,sem_atraso,8fff1cbcebc74fe97c4cb9320a46d2f0,5
4,434d3eec8dd031c322077fb2452e53a0,7a7893d13f178fdf58daeff7347e2d94,delivered,-7,0,sem_atraso,094feeeb71f94c88af5b553084989d40,5
...,...,...,...,...,...,...,...,...
96356,ee15d1d23898663a46ad1c4b08a45759,f0b1375f8ebc3efd01a50b44e47647fe,delivered,-13,0,sem_atraso,61bb221369ffa0c6b0df900f97c5062b,5
96357,0f9458a3e4a303f60f99659d29645ee5,f764be6f129a2ab7f3f8c7d7b15893c8,delivered,-11,0,sem_atraso,a6af20bc9e7fdd6740c3f9f335589bd5,5
96358,2f8f31eb2f7b6572836d662a6625c8e4,cbbb38942ea7e6eaecd622d228fec2b7,delivered,-19,0,sem_atraso,1f13da30e937fb82ac15ca7cc20e4537,5
96359,acbe07f22f29ad7e5a78f30008cc6ec7,b4afeb58ac51bc903c5362286c6a5cfe,delivered,-6,0,sem_atraso,ea1fbd19c48a016b08c92aa1daf658de,5


In [218]:
duckdb.register('data_01', data_01)
print("Tabela registrada")

Tabela registrada


In [ ]:
data_atraso = (data_01.groupby('DtDiasAtraso').agg(media_review=('review_score', 'mean'), qtd_pedidos=('review_score', 'count')).reset_index().sort_values(by='media_review', ascending=False))

data_atraso

,DtDiasAtraso,media_review,qtd_pedidos
3,sem_atraso,4.289980,89944
0,1-3_dias_atraso,3.289871,1856
1,1-7_dias_atraso,2.104214,1756
2,7+_dias_atraso,1.707665,2805


In [ ]:
baseline = data_01[data_01['DtDiasAtraso'] == 'sem_atraso']['review_score'].mean()

media_por_faixa = (data_01.groupby('DtDiasAtraso')['review_score'].mean())

impacto = media_por_faixa - baseline

resultado = (data_01.groupby('DtDiasAtraso').agg(media_review=('review_score', 'mean')))

baseline = resultado.loc['sem_atraso', 'media_review']

resultado['impacto_vs_sem_atraso'] = round(resultado['media_review'] - baseline, 2)
resultado = resultado.sort_values(by='media_review', ascending=False)

resultado['impacto_percentual'] = round((resultado['media_review'] / baseline) - 1, 2)

In [150]:
resultado.reset_index()

,DtDiasAtraso,media_review,impacto_vs_sem_atraso,impacto_percentual
0,sem_atraso,4.289980,0.00,0.00
1,1-3_dias_atraso,3.289871,-1.00,-0.23
2,1-7_dias_atraso,2.104214,-2.19,-0.51
3,7+_dias_atraso,1.707665,-2.58,-0.60


#Preço e frete

In [239]:
query = """
WITH tb_atraso AS (
    SELECT DISTINCT order_id,
            customer_id,
            order_status,
            date(substr(order_delivered_customer_date, 1, 10)) - date(substr(order_estimated_delivery_date, 1, 10)) AS DtDias
    FROM orders
    WHERE order_status = 'delivered'
),

tb_new_atraso AS (

SELECT *,
        CASE
            WHEN DtDias <= 0 THEN 0 
            WHEN DtDias > 0 THEN 1
        END AS DtAtraso,
        CASE
            WHEN DtDias <= 0 THEN 'sem_atraso'
            WHEN DtDias <= 3 THEN '1-3_dias_atraso'
            WHEN DtDias <= 7 THEN '1-7_dias_atraso'
            ELSE '7+_dias_atraso'
        END AS DtDiasAtraso
FROM tb_atraso  
),

tb_preco_frete AS (
SELECT nwa.*,
        oi.price,
        oi.freight_value
FROM tb_new_atraso AS nwa
LEFT JOIN order_items AS oi
ON nwa.order_id = oi.order_id
)

SELECT pf.*,
       ro.review_score,
FROM tb_preco_frete AS pf
LEFT JOIN order_reviews AS ro
ON pf.order_id = ro.order_id



"""

data_02 = duckdb.query(query=query).to_df()

In [230]:
data_02.isnull().sum()

order_id           0
customer_id        0
order_status       0
DtDias             8
DtAtraso           8
DtDiasAtraso       0
price              0
freight_value      0
review_score     827
dtype: int64

In [231]:
preco_frete = data_02.dropna()

In [242]:
duckdb.register('preco_frete', preco_frete)

print("Tabela registrada")

Tabela registrada


In [261]:
query = """

SELECT 
        DtDiasAtraso,
        AVG(price) AS media_preco,
        AVG(freight_value) AS media_frete,
        AVG(review_score) AS media_review_score,
        COUNT(order_id) AS qtd_pedidos

FROM preco_frete
GROUP BY DtDiasAtraso
ORDER BY media_frete DESC


"""

data_03 = duckdb.query(query=query).to_df()

In [260]:
data_03

,DtDiasAtraso,media_preco,media_frete,media_review_score,QtdePedidos
0,7+_dias_atraso,142.579804,24.185453,1.699804,3068
1,1-7_dias_atraso,138.734480,22.140605,2.088672,1951
2,1-3_dias_atraso,119.022437,20.906259,3.228858,2093
3,sem_atraso,118.665635,19.745067,4.207410,102893


In [267]:
baseline_review = 4.207410
baseline_frete = 19.745067

data_03['queda_review'] = round(data_03['media_review_score'] - baseline_review, 2)
data_03['queda_review_%'] = round((data_03['media_review_score'] / baseline_review) - 1, 2)

data_03['aumento_frete'] = round(data_03['media_frete'] - baseline_frete, 2)
data_03['aumento_frete_%'] = round((data_03['media_frete'] / baseline_frete) - 1, 2)

In [268]:
data_03

,DtDiasAtraso,media_preco,media_frete,media_review_score,qtd_pedidos,queda_review,queda_review_%,aumento_frete,aumento_frete_%
0,7+_dias_atraso,142.579804,24.185453,1.699804,3068,-2.51,-0.60,4.44,0.22
1,1-7_dias_atraso,138.734480,22.140605,2.088672,1951,-2.12,-0.50,2.40,0.12
2,1-3_dias_atraso,119.022437,20.906259,3.228858,2093,-0.98,-0.23,1.16,0.06
3,sem_atraso,118.665635,19.745067,4.207410,102893,-0.00,-0.00,-0.00,-0.00
